# AquaInsight — Task 4: Duplicate Detection and Record Linkage

**Internship Project:** Predicting Water Quality Index for Comprehensive Water Assessment

This notebook is one of the nine independent GitHub deliverables. It can be run separately using the supplied water-quality CSV.

## Step-by-step approach

1. Load the supplied dataset.
2. Perform the task-specific analysis.
3. Display quantitative results.
4. Interpret the results for downstream water-quality modeling.
5. Preserve data for review rather than making unsupported automatic corrections.

In [ ]:
# Common setup — AquaInsight Water Quality Internship

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.spatial.distance import mahalanobis

from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, OrdinalEncoder
from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.metrics import mean_squared_error

try:
    from rapidfuzz.fuzz import ratio, token_set_ratio
except ImportError:
    raise ImportError("Install RapidFuzz first: pip install rapidfuzz")

possible_paths = [
    Path("Water Quality(1).csv"),
    Path("Water Quality.csv"),
    Path("../data/Water Quality(1).csv"),
    Path("../data/Water Quality.csv"),
]

DATA_PATH = next((p for p in possible_paths if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Place the supplied CSV beside this notebook.")

df = pd.read_csv(DATA_PATH)

print(f"Dataset: {DATA_PATH}")
print(f"Shape: {df.shape}")
display(df.head())


# Task 4 — Duplicate Detection and Record Linkage

### Internship requirement
Design a duplicate-detection framework using **exact matching, fuzzy string matching (Levenshtein/Jaccard-style similarity), and probabilistic-style record linkage**, followed by review-oriented duplicate flagging.

### Steps
1. Detect exact duplicate rows.
2. Detect duplicate measurement keys.
3. Normalize text fields used for matching.
4. Calculate Levenshtein-style similarity using RapidFuzz.
5. Calculate token/Jaccard similarity for multi-token names.
6. Combine multiple similarity signals into a conservative record-linkage score.
7. Flag candidates for manual verification instead of automatically deleting them.


> **Method note:** The combined linkage score is a practical probabilistic-style candidate score. A formal Fellegi-Sunter probabilistic linkage model would additionally require labeled match/non-match examples or empirically estimated m/u probabilities.

In [10]:
# Step 6 — Exact, fuzzy, and probabilistic-style duplicate candidate detection

from rapidfuzz.fuzz import ratio, token_set_ratio

def normalize_text(value):
    if pd.isna(value):
        return ""
    return " ".join(str(value).lower().strip().split())

def jaccard_similarity(a, b):
    a_tokens = set(normalize_text(a).split())
    b_tokens = set(normalize_text(b).split())
    if not a_tokens and not b_tokens:
        return 1.0
    if not a_tokens or not b_tokens:
        return 0.0
    return len(a_tokens & b_tokens) / len(a_tokens | b_tokens)

# 1. Exact duplicates
exact_duplicates = df[df.duplicated(keep=False)].copy()

# 2. Duplicate measurement keys
key_cols = [
    "MonitoringLocationID", "ActivityStartDate", "ActivityStartTime",
    "CharacteristicName", "ResultValue", "ResultUnit"
]
key_duplicate_count = int(df.duplicated(subset=key_cols, keep=False).sum())

print("Exact full-row duplicates:", len(exact_duplicates))
print("Duplicate candidate rows using measurement key:", key_duplicate_count)

# 3. Conservative fuzzy matching of unique monitoring-location names.
locations = df[["MonitoringLocationID", "MonitoringLocationName"]].drop_duplicates()
location_names = locations["MonitoringLocationName"].dropna().unique().tolist()

fuzzy_candidates = []
for name_a, name_b in itertools.combinations(location_names, 2):
    lev_score = ratio(normalize_text(name_a), normalize_text(name_b)) / 100
    jac_score = jaccard_similarity(name_a, name_b)
    token_score = token_set_ratio(name_a, name_b) / 100

    # A conservative combined linkage score.
    linkage_score = (0.50 * lev_score) + (0.25 * jac_score) + (0.25 * token_score)

    if linkage_score >= 0.90:
        fuzzy_candidates.append({
            "name_a": name_a,
            "name_b": name_b,
            "levenshtein_similarity": round(lev_score, 3),
            "jaccard_similarity": round(jac_score, 3),
            "token_similarity": round(token_score, 3),
            "linkage_score": round(linkage_score, 3)
        })

fuzzy_candidates_df = pd.DataFrame(fuzzy_candidates).sort_values(
    "linkage_score", ascending=False
)

display(fuzzy_candidates_df.head(20))
print("Fuzzy candidates flagged for manual review:", len(fuzzy_candidates_df))


   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ------------- -------------------------- 0.5/1.6 MB 5.3 MB/s eta 0:00:01
   -------------------------------- ------- 1.3/1.6 MB 4.0 MB/s eta 0:00:01
   ---------------------------------------- 1.6/1.6 MB 3.9 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Exact full-row duplicates: 0
Duplicate candidate rows using measurement key: 382


([('BIG DAM WEST LAKE IN MIDDLE OF LAKE',
   'BIG DAM EAST LAKE IN MIDDLE OF LAKE',
   96.66666666666667),
  ('LAKE OF ISLANDS CAPE BRETON HIGHLANDS NATIONAL PARK OF CANADA',
   'WARREN LAKE 100 M SOUTH POINT CAPE BRETON HIGHLANDS NATIONAL PARK OF CANADA',
   92.5925925925926),
  ('LAKE OF ISLANDS CAPE BRETON HIGHLANDS NATIONAL PARK OF CANADA',
   'GLASGOW LAKE, CAPE BRETON HIGHLANDS NATIONAL PARK OF CANADA',
   92.3076923076923),
  ('LAKE OF ISLANDS CAPE BRETON HIGHLANDS NATIONAL PARK OF CANADA',
   'FRENCH LAKE CAPE BRETON HIGHLANDS NATIONAL PARK OF CANADA',
   93.45794392523365),
  ('ROUND LAKE, CAPE BRETON HIGHLANDS NATIONAL PARK OF CANADA',
   'GLASGOW LAKE, CAPE BRETON HIGHLANDS NATIONAL PARK OF CANADA',
   94.44444444444444),
  ('WARREN LAKE 100 M SOUTH POINT CAPE BRETON HIGHLANDS NATIONAL PARK OF CANADA',
   'FRENCH LAKE CAPE BRETON HIGHLANDS NATIONAL PARK OF CANADA',
   93.45794392523365)],
 6)

## Task 4 — Conclusion

The analysis above completes the requested **Duplicate Detection and Record Linkage** component of the AquaInsight internship assignment. Results should be interpreted together with domain requirements and the official WQI definition when it becomes available.